# VocabBook 中 FSRS_v4 算法复刻过程和验证

## 算法中用到的符号
- $R$：可检索性（回忆概率）
- $S$：稳定性（对应 $R = 90\%$ 时的间隔）
  - $S_r$：回忆后新的稳定性
  - $S_f$：遗忘后新的稳定性
- $D$：难度（$D \in [1, 10]$）
- $G$：评分（Anki 中的评分等级）
  - $1$: 再次
  - $2$: 困难
  - $3$: 良好
  - $4$: 容易


In [1]:
from enum import IntEnum

class FSRSReviewRating(IntEnum):
    AGAIN = 1
    HARD = 2
    GOOD = 3
    EASY = 4

# 用户不给定初始值则赋默认值
FSRS_DEFAULT_RATING = FSRSReviewRating.HARD

## 默认参数
[0.4, 0.6, 2.4, 5.8, 4.93, 0.94, 0.86, 0.01, 1.49, 0.14, 0.94, 2.18, 0.05, 0.34, 1.26, 0.29, 2.61]

In [2]:
FSRS_V4_DEFAULT_PARAMS = [
    0.4, #w_1
    0.6, #w_2
    2.4, #w_3
    5.8, #w_4
    4.93, #w_5
    0.94, #w_6
    0.86, #w_7
    0.01, #w_8
    1.49, #w_9
    0.14, #w_10
    0.94, #w_11
    2.18, #w_12
    0.05, #w_13
    0.34, #w_14
    1.26, #w_15
    0.29, #w_16
    2.61 #w_17
]

## 公式说明

> $W_i$ 表示 $W[i]$。该版本使用 17 个参数。记忆状态由稳定性（S）和难度（D）共同表示。

---

首次评分后的初始稳定性：

$$S_0(G) = w_{G-1}.$$

例如：$S_0(1) = w_0$ 是第一次评分为 Again（再次）时的初始稳定性。第一次评分为 Easy（容易）时，初始稳定性为 $S_0(4) = w_3$。

In [3]:
def calculate_initial_stability(rating: FSRSReviewRating = FSRS_DEFAULT_RATING) -> float:
    """计算首次评分后的初始稳定性"""
    w = FSRS_V4_DEFAULT_PARAMS
    initial_stability = w[rating.value - 1]
    return initial_stability

print(f"默认评分（回忆困难）的初始稳定性：{calculate_initial_stability()}") # 输出：0.6
print(f"完全不认识的初始稳定性：{calculate_initial_stability(FSRSReviewRating.AGAIN)}") # 输出：0.4
print(f"认识的初始稳定性：{calculate_initial_stability(FSRSReviewRating.GOOD)}") # 输出：2.4
print(f"完全掌握的初始稳定性：{calculate_initial_stability(FSRSReviewRating.EASY)}") # 输出：5.8

默认评分（回忆困难）的初始稳定性：0.6
完全不认识的初始稳定性：0.4
认识的初始稳定性：2.4
完全掌握的初始稳定性：5.8


---

首次评分后的初始难度：

$$D_0(G) = w_4 - (G - 3) \cdot w_5.$$

其中，第一次评分为 Good（良好）时，$D_0(3) = w_4$。

In [4]:
def calculate_initial_difficulty(rating: FSRSReviewRating = FSRS_DEFAULT_RATING) -> float:
    """首次评分后的初始难度"""
    w = FSRS_V4_DEFAULT_PARAMS
    w4 = w[4]
    w5 = w[5]
    G = rating.value
    initial_difficulty = w4 - (G - 3) * w5
    initial_difficulty = max(1.0, min(10.0, initial_difficulty)) # 难度要求在1~10之间
    return round(initial_difficulty, 2)

# 测试1：默认评分（HARD=2）
print("=== 默认评分（回忆困难/HARD=2）===")
print(f"初始稳定性：{calculate_initial_stability()}") # 预期：0.6
print(f"初始难度：{calculate_initial_difficulty()}") # 预期：4.93 - (2-3)*0.94 = 5.87
# 测试2：AGAIN=1（完全不认识）
print("\n=== 完全不认识/AGAIN=1 ===")
print(f"初始稳定性：{calculate_initial_stability(FSRSReviewRating.AGAIN)}") # 预期：0.4
print(f"初始难度：{calculate_initial_difficulty(FSRSReviewRating.AGAIN)}") # 预期：4.93 - (1-3)*0.94 = 6.81
# 测试3：GOOD=3（认识）
print("\n=== 认识/GOOD=3 ===")
print(f"初始稳定性：{calculate_initial_stability(FSRSReviewRating.GOOD)}") # 预期：2.4
print(f"初始难度：{calculate_initial_difficulty(FSRSReviewRating.GOOD)}")  # 预期：4.93（因为(3-3)=0）
# 测试4：EASY=4（完全掌握）
print("\n=== 完全掌握/EASY=4 ===")
print(f"初始稳定性：{calculate_initial_stability(FSRSReviewRating.EASY)}") # 预期：5.8
print(f"初始难度：{calculate_initial_difficulty(FSRSReviewRating.EASY)}") # 预期：4.93 - (4-3)*0.94 = 3.99

=== 默认评分（回忆困难/HARD=2）===
初始稳定性：0.6
初始难度：5.87

=== 完全不认识/AGAIN=1 ===
初始稳定性：0.4
初始难度：6.81

=== 认识/GOOD=3 ===
初始稳定性：2.4
初始难度：4.93

=== 完全掌握/EASY=4 ===
初始稳定性：5.8
初始难度：3.99


---

复习后的新难度：

$$D'(D, G) = w_7 \cdot D_0(3) + (1 - w_7) \cdot \bigl(D - w_6 \cdot (G - 3)\bigr).$$

它会先用 $D' = D - w_6 \cdot (G - 3)$ 计算临时难度，再通过均值回归 $w_7 \cdot D_0(3) + (1 - w_7) \cdot D'$ 进行修正，以避免出现“难度地狱（ease hell）”。

In [5]:
def calculate_updated_difficulty(current_difficulty: float, rating: FSRSReviewRating, w=FSRS_V4_DEFAULT_PARAMS) -> float:
    """
    计算复习后的新难度

    :param current_difficulty: 当前难度
    :param rating: 本次复习的评分等级
    :param w: 参数
    :return: 新难度 2位小数 取值范围1到10
    """
    w6 = w[6]
    w7 = w[7]
    d0_3 = w[4]
    # 计算临时难度
    G = rating.value
    temp_d = current_difficulty - w6 * (G - 3)
    # 均值回归修正和取值范围
    d_prime = w7 * d0_3 + (1 - w7) * temp_d
    d_prime = max(1.0, min(10.0, d_prime))
    return round(d_prime, 2)

# 测试场景：单词当前难度D=5.87（之前HARD评分的初始难度），本次复习评GOOD
current_d = 5.87
rating = FSRSReviewRating.GOOD
new_d = calculate_updated_difficulty(current_d, rating)
print(f"测试1：当前难度={current_d}，复习评分=GOOD → 新难度={new_d}")
# temp_d = 5.87 - 0.86*(3-3) = 5.87
# d_prime = 0.01*4.93 + 0.99*5.87 = 0.0493 + 5.8113 = 5.8606 → 保留2位=5.86

# 测试场景2：当前难度D=5.87，本次复习评EASY
rating2 = FSRSReviewRating.EASY
new_d2 = calculate_updated_difficulty(current_d, rating2)
print(f"测试2：当前难度={current_d}，复习评分=EASY → 新难度={new_d2}")
# temp_d = 5.87 - 0.86*(4-3) = 5.87 - 0.86 = 5.01
# d_prime = 0.01*4.93 + 0.99*5.01 = 0.0493 + 4.9599 = 5.0092 → 保留2位=5.01

# 测试场景3：当前难度D=3.99，本次复习评HARD
current_d3 = 3.99
rating3 = FSRSReviewRating.HARD
new_d3 = calculate_updated_difficulty(current_d3, rating3)
print(f"测试3：当前难度={current_d3}，复习评分=HARD → 新难度={new_d3}")
# temp_d = 3.99 - 0.86*(2-3) = 3.99 + 0.86 = 4.85
# d_prime = 0.01*4.93 + 0.99*4.85 = 0.0493 + 4.8015 = 4.8508 → 保留2位=4.85

测试1：当前难度=5.87，复习评分=GOOD → 新难度=5.86
测试2：当前难度=5.87，复习评分=EASY → 新难度=5.01
测试3：当前难度=3.99，复习评分=HARD → 新难度=4.85


---

自上次复习 t 天后的可检索性:

$$R(t, S) = \left(1 + \frac{t}{9 \cdot S}\right)^{-1},$$

其中，当 $t = S$ 时，$R(t, S) = 0.9$。

将目标回忆率代入上面公式中的 $R$ 并求解 $t$，即可得到下一次复习间隔：

$$I(r, S) = 9 \cdot S \cdot \left(\frac{1}{r} - 1\right),$$

其中：当 $r = 0.9$ 时，$I(r, S) = S$。

In [6]:
def calculate_retrievability(days_since_review: float, stability: float) -> float:
    """
    计算上次复习t天后的可见索性

    :param days_since_review: 自上次复习的天数
    :param stability: 单词当前稳定性
    :return: 可检索性
    """
    stability = max(0.01, stability) # 避免除以0（稳定性最小为0.01）
    denominator = 1 + (days_since_review / (9 * stability))
    retrievability = 1 / denominator
    return round(retrievability, 3)

def calculate_review_interval(stability: float, target_recall: float = 0.9) -> float:
    """
    计算下次复习间隔

    :param stability: 当前单词稳定性
    :param target_recall: 目标回忆率
    :return: 下次复习间隔（天 2位小数）
    """
    target_recall = max(0.1, min(0.99, target_recall)) # 避免目标回忆率非法
    interval = 9 * stability * (1 / target_recall - 1)
    interval = max(0.1, interval) # 间隔最小为0.1天
    return round(interval, 2)

# 测试1：可检索性核心特性（t=S时，R=0.9）
t = 2.4  # 天数
S = 2.4  # 稳定性
R = calculate_retrievability(t, S)
print(f"测试1：t={t}, S={t} → 可检索性R={R}（预期0.9）")
# 测试2：复习间隔核心特性（r=0.9时，I=S）
S = 2.4
I = calculate_review_interval(S, 0.9)
print(f"测试2：S={S}, r=0.9 → 复习间隔I={I}（预期2.4）")
# 测试3：实际场景（稳定性=5.8，目标回忆率0.9）
S3 = 5.8
I3 = calculate_review_interval(S3)
R3 = calculate_retrievability(I3, S3)  # 间隔=稳定性，R=0.9
print(f"测试3：稳定性={S3} → 复习间隔={I3}天，此时可检索性={R3}")
# 测试4：自定义目标回忆率（r=0.8，更宽松，间隔更长）
S4 = 2.4
I4 = calculate_review_interval(S4, 0.8)
print(f"测试4：S={S4}, r=0.8 → 复习间隔I={I4}（预期9*2.4*(1/0.8-1)=5.4）")

测试1：t=2.4, S=2.4 → 可检索性R=0.9（预期0.9）
测试2：S=2.4, r=0.9 → 复习间隔I=2.4（预期2.4）
测试3：稳定性=5.8 → 复习间隔=5.8天，此时可检索性=0.9
测试4：S=2.4, r=0.8 → 复习间隔I=5.4（预期9*2.4*(1/0.8-1)=5.4）
